In [0]:
#import all required  libraries
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
#create SparkSession object for interacting with master node
spark=SparkSession.builder.appName("bank-project").getOrCreate()

In [0]:
#read accounts data from csv files
accounts_df=spark.read.format("csv")\
    .option("inferSchema",True)\
        .option("header",True)\
            .load("/Workspace/Users/sudhakarkuruba000@gmail.com/notebbooksrepo/source_csv-files/accounts.csv")

#read atm_transactions data from csv files
atm_df=spark.read.format("csv")\
    .option("header",True)\
        .option("inferSchema",True)\
            .load("/Workspace/Users/sudhakarkuruba000@gmail.com/notebbooksrepo/source_csv-files/atm_transactions.csv")

#read branches data from csv files
branch_df=spark.read.format("csv")\
    .option("header",True)\
        .option("inferSchema",True)\
            .load("/Workspace/Users/sudhakarkuruba000@gmail.com/notebbooksrepo/source_csv-files/branches.csv")

#read credit_cards data from csv files
credit_df=spark.read.format("csv")\
    .option("header",True)\
        .option("inferSchema",True)\
            .load("/Workspace/Users/sudhakarkuruba000@gmail.com/notebbooksrepo/source_csv-files/credit_cards.csv")


#read customers data from csv files
customer_df=spark.read.format("csv")\
    .option("header",True)\
        .option("inferSchema",True)\
            .load("/Workspace/Users/sudhakarkuruba000@gmail.com/notebbooksrepo/source_csv-files/customers.csv")

#read employees data from csv files
employees_df=spark.read.format("csv")\
    .option("header",True)\
        .option("inferSchema",True)\
            .load("/Workspace/Users/sudhakarkuruba000@gmail.com/notebbooksrepo/source_csv-files/employees.csv")


In [0]:
#write to bronze layer same data as from source
accounts_df.write.mode("overwrite").saveAsTable("bank.bronze.accounts")
atm_df.write.format("delta").mode("overwrite").saveAsTable("bank.bronze.atm_transactions")
branch_df.write.format("delta").mode("overwrite").saveAsTable("bank.bronze.branches")
credit_df.write.format("delta").mode("overwrite").saveAsTable("bank.bronze.credit_cards")
customer_df.write.format("delta").mode("overwrite").saveAsTable("bank.bronze.customers")
employees_df.write.format("delta").mode("overwrite").saveAsTable("bank.bronze.employees")


In [0]:
#read data from bronze layer to dataframes and perform transformations and write to silver layer
b_accounts_df=spark.table("bank.bronze.accounts")
b_atm_df=spark.table("bank.bronze.atm_transactions")
b_branch_df=spark.table("bank.bronze.branches")
b_credit_df=spark.table("bank.bronze.credit_cards")
b_customer_df=spark.table("bank.bronze.customers")
b_employees_df=spark.table("bank.bronze.employees")

In [0]:
#perform transformations on accounts dataframe
s_accounts=b_accounts_df\
    .filter(col("account_id").isNotNull())\
        .withColumn("amount",coalesce(col("amount"),lit(0.00)))\
            .withColumn("status",when(col("status").isin(["ACTIVE","INACTIVE","PENDING"]),col("status")).otherwise(lit("NO_STATUS")))\
                .withColumn("type",when(col("type").isin(["SAVINGS","CURRENT","LOAN"]),col("type")).otherwise(lit("NO_TYPE")))\
                    .withColumn("date",to_date(col("date"),"MM-dd-yyyy"))\
                        .withColumn("timestamp",current_timestamp())
s1_accounts=s_accounts.dropDuplicates(["account_id"])
s_accounts.write.mode("overwrite").saveAsTable("bank.silver.accounts")
#perform transformations on atm_transactions dataframe